In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from pathlib import Path
import matplotlib.pyplot as plt

In [ ]:
class RRDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = Path(root_dir)
        self.transform = transform
        self.samples = []
        
        self.class_ai_to_idx = {"real": 0, "ai": 1}
        self.class_domain_to_idx = {"original": 0, "redigital": 1, "transfer": 2}

        for domain in self.class_domain_to_idx.keys():
            domain_path = self.root_dir / domain
            if not domain_path.exists(): continue
                
            for class_name in self.class_ai_to_idx.keys():
                class_path = domain_path / class_name
                if not class_path.exists(): continue
                for pattern in ("*.jpg", "*.png"):
                    for img_path in class_path.glob(pattern):
                        self.samples.append((img_path, self.class_ai_to_idx[class_name], self.class_domain_to_idx[domain]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label_ai, label_domain = self.samples[idx]
        image = Image.open(img_path).convert("RGB")
        
        if self.transform:
            image = self.transform(image)
            
        return image, {
            'label_ai': torch.tensor(label_ai, dtype=torch.long),
            'label_domain': torch.tensor(label_domain, dtype=torch.long)
        }

In [ ]:
class RGBNetInput(nn.Module):
    def __init__(self):
        super(RGBNetInput, self).__init__()
        # Eventuali pre-processing specifici per RGB (es. filtri di denoising)
        # potrebbero essere inseriti qui. Per ora, lasciamo l'input puro.

    def forward(self, x):
        # L'input x è già il tensore RGB [B, 3, H, W]
        return x

In [ ]:
# Setup Dataloader
my_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

dataset = RRDataset(root_dir=Path(".")/"RRDataset_final", transform=my_transforms)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

# Visualizzazione
images, labels = next(iter(dataloader))
index = 0

# Ripristiniamo i canali per la visualizzazione (H, W, C)
img_display = images[index].permute(1, 2, 0).numpy()
# Denormalizzazione rapida per vedere i colori originali
img_display = (img_display * [0.229, 0.224, 0.225]) + [0.485, 0.456, 0.406]

plt.imshow(img_display.clip(0, 1))
plt.title(f"AI: {labels['label_ai'][index].item()} | Domain: {labels['label_domain'][index].item()}")
plt.axis('off')
plt.show()

In [ ]:
class RGBNet(nn.Module):
    def __init__(self, feature_dim=64):
        super(RGBNet, self).__init__()

        self.input_layer = RGBNetInput()
        
        # Architettura Convoluzionale per estrazione feature spaziali
        self.layers = nn.Sequential(
            self._make_layer(3, 64),   # Input: 3 canali RGB
            self._make_layer(64, 128), 
            self._make_layer(128, 256) # Output finale: 256 canali
        )
        
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        
        # Strato per generare l'embedding finale
        self.fc_embedding = nn.Sequential(
            nn.Linear(256, feature_dim),
            nn.ReLU(),
            nn.Dropout(0.3)
        )

    def _make_layer(self, in_channels, out_channels):
        return nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

    def forward(self, x):
        # 1. Pre-processing
        x = self.input_layer(x)
        
        # 2. Estrazione feature
        x = self.layers(x)
        
        # 3. Pooling e Flattening
        x = self.global_pool(x)
        x = torch.flatten(x, 1) # Risultato: [B, 256]
        
        # 4. Creazione Embedding
        embedding = self.fc_embedding(x)
        
        return embedding